# Notebook 2: FiLM World Model — Training

Implements the full FiLM-based commodity market prediction model from `model_architecture_v1_FiLM.md`.

- **Section A**: Config, data pipeline, dataset
- **Section B**: Model architecture (all modules)
- **Section C**: Stage 1 training (frozen Qwen, cached embeddings)
- **Section D**: Stage 2 training (LoRA fine-tuning)

**Environment**: Kaggle GPU. `HF_token` assumed available.

In [ ]:
!pip install -q transformers accelerate peft

---
## Section A: Config & Data Pipeline

In [ ]:
import os
import math
from bisect import bisect_left, bisect_right
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

In [ ]:
# ============================================================
# Configuration
# ============================================================

@dataclass
class Config:
    # Data paths (Kaggle)
    data_dir: str = "/kaggle/input/finance-world-model-dataset"
    embeddings_path: str = "/kaggle/input/event-embeddings/event_embeddings.pt"

    # Architecture
    n_features: int = 46
    d_model: int = 256
    lookback: int = 10
    horizon: int = 10
    n_transformer_layers: int = 2
    n_heads: int = 4
    ff_dim: int = 1024
    dropout: float = 0.1
    max_events_per_window: int = 8

    # Qwen
    qwen_model_id: str = "Qwen/Qwen3-4B"
    qwen_hidden: int = 2560
    lora_r: int = 16
    lora_alpha: int = 32
    lora_target_modules: tuple = ("q_proj", "v_proj")

    # Critical features for loss
    critical_features: tuple = (
        "YF_Price_Crude Oil (WTI)",
        "YF_Price_Crude Oil (Brent)",
        "YF_Price_Gold",
        "YF_Price_Natural Gas",
        "YF_Price_Silver",
        "YF_Price_Copper",
    )

    # Features using simple difference instead of log return (can be zero)
    diff_features: tuple = ("FRED_DFF", "FRED_DGS10", "FRED_T10YIE")

    # Market context for event text formatting
    context_map: dict = None  # set in __post_init__

    # Data splits
    train_start: str = "2015-01-01"
    train_end: str = "2023-12-31"
    val_start: str = "2024-01-01"
    val_end: str = "2024-12-31"
    test_start: str = "2025-01-01"
    test_end: str = "2025-12-31"

    # Stage 1 training
    stage1_batch_size: int = 64
    stage1_lr: float = 1e-3
    stage1_weight_decay: float = 1e-2
    stage1_epochs: int = 150

    # Stage 2 training
    stage2_batch_size: int = 8
    stage2_lr_market: float = 1e-4
    stage2_lr_lora: float = 1e-5
    stage2_weight_decay: float = 1e-2
    stage2_epochs: int = 30
    stage2_grad_accum: int = 4

    def __post_init__(self):
        self.context_map = {
            "YF_Price_Crude Oil (WTI)": ("WTI", "${:.2f}"),
            "YF_Price_Crude Oil (Brent)": ("Brent", "${:.2f}"),
            "YF_Price_Gold": ("Gold", "${:.0f}"),
            "YF_Price_Natural Gas": ("NatGas", "${:.2f}"),
            "FRED_DGS10": ("10Y", "{:.2f}%"),
            "FRED_DTWEXBGS": ("DXY", "{:.1f}"),
            "FRED_SP500": ("S&P500", "{:.0f}"),
            "FRED_VIXCLS": ("VIX", "{:.1f}"),
        }


cfg = Config()
print(f"Config: d_model={cfg.d_model}, layers={cfg.n_transformer_layers}, heads={cfg.n_heads}")

### A.1 Preprocessing: Log Returns & Event Matching

In [ ]:
# ============================================================
# Load and preprocess market data
# ============================================================

market_df = pd.read_csv(
    f"{cfg.data_dir}/combined_commodity_data.csv",
    parse_dates=["date"],
    index_col="date",
)
feature_cols = market_df.columns.tolist()
assert len(feature_cols) == cfg.n_features, f"Expected {cfg.n_features} features, got {len(feature_cols)}"

# Compute critical feature indices
critical_indices = [feature_cols.index(f) for f in cfg.critical_features]
print(f"Critical feature indices: {list(zip(cfg.critical_features, critical_indices))}")

# Compute log returns (or simple diff for rate features)
diff_cols = [c for c in feature_cols if c in cfg.diff_features]
log_cols = [c for c in feature_cols if c not in cfg.diff_features]

deltas_df = pd.DataFrame(index=market_df.index[1:], columns=feature_cols, dtype=np.float32)

# Log returns for price/index features
for col in log_cols:
    vals = market_df[col].values.astype(np.float64)
    # Avoid log(0): replace zeros with previous value
    vals[vals == 0] = np.nan
    vals = pd.Series(vals).ffill().bfill().values
    deltas_df[col] = np.diff(np.log(vals)).astype(np.float32)

# Simple differences for rate features
for col in diff_cols:
    deltas_df[col] = np.diff(market_df[col].values).astype(np.float32)

# Clip extreme values
deltas_df = deltas_df.clip(-0.5, 0.5)

# Fill any remaining NaNs with 0
deltas_df = deltas_df.fillna(0.0)

print(f"Log returns shape: {deltas_df.shape}")
print(f"Date range: {deltas_df.index[0]} to {deltas_df.index[-1]}")
deltas_df.describe().loc[["mean", "std", "min", "max"]]

In [ ]:
# ============================================================
# Load events and build date-sorted index for fast matching
# ============================================================

events_df = pd.read_csv(f"{cfg.data_dir}/gdelt_events_with_market_impact.csv")
events_df["event_time"] = pd.to_datetime(events_df["event_time"])
events_df = events_df[events_df["market_impact"] == 1].reset_index(drop=True)
print(f"Market-impact events: {len(events_df)}")

# Sorted event dates for bisect-based matching
event_dates_sorted = events_df["event_time"].sort_values().values
event_idx_by_date = events_df.sort_values("event_time").index.values
# Convert to list of timestamps for bisect
event_timestamps = [pd.Timestamp(d) for d in event_dates_sorted]


def find_events_in_window(start_date, end_date):
    """Find indices of events falling within [start_date, end_date] using bisect."""
    left = bisect_left(event_timestamps, pd.Timestamp(start_date))
    right = bisect_right(event_timestamps, pd.Timestamp(end_date))
    return event_idx_by_date[left:right].tolist()


# Quick test
test_events = find_events_in_window("2020-01-01", "2020-01-10")
print(f"Events in 2020-01-01 to 2020-01-10: {len(test_events)} events")

### A.2 Dataset

In [ ]:
# ============================================================
# PyTorch Dataset
# ============================================================

class CommodityEventDataset(Dataset):
    """
    Sliding window dataset for the FiLM world model.

    Each sample:
      - market_deltas: (lookback, n_features) log returns
      - anchor_state: (n_features,) raw values at end of input window
      - target_deltas: (horizon, n_features) log returns to predict
      - event_embeds: (max_events, qwen_hidden) pre-computed [cached mode]
      - event_mask: (max_events,) True = padded/absent
      - n_events: int
    """

    def __init__(self, deltas_df, raw_df, events_df, cfg,
                 start_date, end_date, embedding_cache=None):
        self.cfg = cfg
        self.embedding_cache = embedding_cache  # dict: event_idx -> tensor(qwen_hidden)

        # Filter to date range
        mask = (deltas_df.index >= start_date) & (deltas_df.index <= end_date)
        self.deltas = deltas_df[mask].values.astype(np.float32)  # (T, n_features)
        self.dates = deltas_df[mask].index

        # Raw values for anchor state (aligned: raw_df has one extra row at the start)
        raw_mask = (raw_df.index >= start_date) & (raw_df.index <= end_date)
        self.raw_values = raw_df[raw_mask].values.astype(np.float32)
        self.raw_dates = raw_df[raw_mask].index

        self.events_df = events_df

        # Build valid sample indices
        total_needed = cfg.lookback + cfg.horizon
        self.n_samples = max(0, len(self.deltas) - total_needed + 1)
        print(f"Dataset [{start_date} to {end_date}]: {self.n_samples} samples")

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        L = self.cfg.lookback
        H = self.cfg.horizon

        # Input window: deltas[idx : idx+L]
        market_deltas = torch.from_numpy(self.deltas[idx : idx + L])

        # Target window: deltas[idx+L : idx+L+H]
        target_deltas = torch.from_numpy(self.deltas[idx + L : idx + L + H])

        # Anchor state: raw values at end of input window (idx+L-1 in deltas corresponds
        # to the same date in raw_df)
        anchor_date = self.dates[idx + L - 1]
        # Find matching raw date
        raw_idx = self.raw_dates.get_loc(anchor_date)
        anchor_state = torch.from_numpy(self.raw_values[raw_idx])

        # Find events in input window
        window_start = self.dates[idx]
        window_end = self.dates[idx + L - 1]
        event_indices = find_events_in_window(window_start, window_end)

        # Cap events
        event_indices = event_indices[: self.cfg.max_events_per_window]
        n_events = len(event_indices)

        # Build event embeddings (cached mode)
        max_k = self.cfg.max_events_per_window
        if self.embedding_cache is not None:
            event_embeds = torch.zeros(max_k, self.cfg.qwen_hidden)
            for i, eidx in enumerate(event_indices):
                event_embeds[i] = self.embedding_cache[eidx]
        else:
            event_embeds = torch.zeros(max_k, self.cfg.qwen_hidden)  # placeholder

        # Event mask: True = padded (to be masked in attention)
        event_mask = torch.ones(max_k, dtype=torch.bool)
        event_mask[:n_events] = False

        return {
            "market_deltas": market_deltas,      # (L, n_features)
            "anchor_state": anchor_state,          # (n_features,)
            "target_deltas": target_deltas,        # (H, n_features)
            "event_embeds": event_embeds,          # (max_k, qwen_hidden)
            "event_mask": event_mask,              # (max_k,)
            "n_events": n_events,
            "event_indices": event_indices,        # list[int], variable length
        }

In [ ]:
# ============================================================
# Load embedding cache and create datasets
# ============================================================

cache_data = torch.load(cfg.embeddings_path, weights_only=False)
embedding_cache = cache_data["embeddings"]
print(f"Loaded {len(embedding_cache)} cached embeddings (dim={cache_data['hidden_size']})")

# Verify hidden size matches config
assert cache_data["hidden_size"] == cfg.qwen_hidden, (
    f"Cache hidden={cache_data['hidden_size']} != config={cfg.qwen_hidden}. "
    f"Update cfg.qwen_hidden."
)

train_ds = CommodityEventDataset(
    deltas_df, market_df, events_df, cfg,
    cfg.train_start, cfg.train_end, embedding_cache,
)
val_ds = CommodityEventDataset(
    deltas_df, market_df, events_df, cfg,
    cfg.val_start, cfg.val_end, embedding_cache,
)

# Quick sanity check
sample = train_ds[0]
for k, v in sample.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k}: {v.shape} {v.dtype}")
    else:
        print(f"  {k}: {v}")

In [ ]:
def collate_fn(batch):
    """Stack tensors, drop variable-length event_indices."""
    return {
        "market_deltas": torch.stack([s["market_deltas"] for s in batch]),
        "anchor_state": torch.stack([s["anchor_state"] for s in batch]),
        "target_deltas": torch.stack([s["target_deltas"] for s in batch]),
        "event_embeds": torch.stack([s["event_embeds"] for s in batch]),
        "event_mask": torch.stack([s["event_mask"] for s in batch]),
    }


train_loader = DataLoader(
    train_ds, batch_size=cfg.stage1_batch_size, shuffle=True,
    collate_fn=collate_fn, num_workers=2, pin_memory=True,
)
val_loader = DataLoader(
    val_ds, batch_size=cfg.stage1_batch_size, shuffle=False,
    collate_fn=collate_fn, num_workers=2, pin_memory=True,
)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

---
## Section B: Model Architecture

In [ ]:
# ============================================================
# B.1 Market MLP
# ============================================================

class MarketMLP(nn.Module):
    """Per-timestep projection of market deltas into model dimension."""

    def __init__(self, n_features, d_model):
        super().__init__()
        self.linear = nn.Linear(n_features, d_model)
        self.act = nn.GELU()

    def forward(self, x):
        # x: (B, L, n_features) -> (B, L, d_model)
        return self.act(self.linear(x))

In [ ]:
# ============================================================
# B.2 FiLM Layer
# ============================================================

class FiLMLayer(nn.Module):
    """Feature-wise Linear Modulation. Conditioning signal generates
    gamma (scale) and beta (shift) applied uniformly across timesteps."""

    def __init__(self, conditioning_dim, feature_dim):
        super().__init__()
        self.gamma_linear = nn.Linear(conditioning_dim, feature_dim)
        self.beta_linear = nn.Linear(conditioning_dim, feature_dim)

        # Initialize as identity: gamma=1, beta=0
        nn.init.ones_(self.gamma_linear.bias)
        nn.init.zeros_(self.gamma_linear.weight)
        nn.init.zeros_(self.beta_linear.bias)
        nn.init.zeros_(self.beta_linear.weight)

    def forward(self, x, conditioning):
        # x: (B, L, d)  conditioning: (B, cond_dim)
        gamma = self.gamma_linear(conditioning).unsqueeze(1)  # (B, 1, d)
        beta = self.beta_linear(conditioning).unsqueeze(1)    # (B, 1, d)
        return gamma * x + beta

In [ ]:
# ============================================================
# B.3 Event Encoder (Cached Mode for Stage 1)
# ============================================================

class EventEncoder(nn.Module):
    """Projects pre-computed Qwen embeddings to model dimension.
    Produces per-event representations and a global pool."""

    def __init__(self, qwen_hidden, d_model):
        super().__init__()
        self.projection = nn.Linear(qwen_hidden, d_model)
        # Learned embedding for windows with no events
        self.no_event_embed = nn.Parameter(torch.zeros(d_model))

    def forward(self, event_embeds, event_mask):
        """
        Args:
            event_embeds: (B, max_k, qwen_hidden) pre-computed
            event_mask: (B, max_k) True = padded/absent
        Returns:
            e_global: (B, d_model) pooled event representation
            e_reprs: (B, max_k, d_model) per-event representations
        """
        B = event_embeds.shape[0]

        # Project all event embeddings
        e_reprs = self.projection(event_embeds)  # (B, max_k, d_model)

        # Mean pool over valid events
        valid_mask = ~event_mask  # True = valid event
        n_valid = valid_mask.sum(dim=1, keepdim=True).clamp(min=1)  # (B, 1)

        # Zero out padded positions before pooling
        masked_reprs = e_reprs * valid_mask.unsqueeze(-1).float()
        e_global = masked_reprs.sum(dim=1) / n_valid.float()  # (B, d_model)

        # For samples with zero events, use learned no-event embedding
        no_events = valid_mask.sum(dim=1) == 0  # (B,)
        if no_events.any():
            e_global[no_events] = self.no_event_embed.unsqueeze(0)

        return e_global, e_reprs

In [ ]:
# ============================================================
# B.4 Cross-Attention: Event -> Market
# ============================================================

class EventMarketCrossAttention(nn.Module):
    """Market sequence attends to event representations.
    Information flows event -> market only."""

    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        self.cross_attn = nn.MultiheadAttention(
            d_model, n_heads, dropout=dropout, batch_first=True,
        )
        self.norm = nn.LayerNorm(d_model)

    def forward(self, market_seq, event_reprs, event_mask):
        """
        Args:
            market_seq: (B, L, d) query
            event_reprs: (B, max_k, d) key/value
            event_mask: (B, max_k) True = padded
        Returns:
            (B, L, d) modulated market sequence
        """
        attn_out, _ = self.cross_attn(
            query=market_seq,
            key=event_reprs,
            value=event_reprs,
            key_padding_mask=event_mask,
        )
        # Residual + LayerNorm
        return self.norm(market_seq + attn_out)

In [ ]:
# ============================================================
# B.5 Temporal Transformer
# ============================================================

class TemporalTransformer(nn.Module):
    """Transformer encoder over the temporal market sequence."""

    def __init__(self, d_model, n_heads, n_layers, ff_dim, dropout, seq_len):
        super().__init__()
        self.pos_encoding = nn.Parameter(torch.randn(1, seq_len, d_model) * 0.02)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=ff_dim,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

    def forward(self, x):
        # x: (B, L, d)
        x = x + self.pos_encoding
        return self.encoder(x)

In [ ]:
# ============================================================
# B.6 Prediction Head
# ============================================================

class PredictionHead(nn.Module):
    """Projects hidden states to predicted deltas.
    Loss is MSE on critical features only."""

    def __init__(self, d_model, n_features, critical_indices):
        super().__init__()
        self.head = nn.Linear(d_model, n_features)
        self.register_buffer(
            "critical_indices",
            torch.tensor(critical_indices, dtype=torch.long),
        )

    def forward(self, h, target_deltas=None):
        """
        Args:
            h: (B, H, d_model)
            target_deltas: (B, H, n_features) or None
        Returns:
            predicted: (B, H, n_features)
            loss: scalar or None
        """
        predicted = self.head(h)  # (B, H, n_features)

        loss = None
        if target_deltas is not None:
            pred_critical = predicted[:, :, self.critical_indices]
            target_critical = target_deltas[:, :, self.critical_indices]
            loss = F.mse_loss(pred_critical, target_critical)

        return predicted, loss

In [ ]:
# ============================================================
# B.7 FiLM World Model — Top Level
# ============================================================

class FiLMWorldModel(nn.Module):
    """Full FiLM-based commodity market prediction model.

    Architecture flow:
      MarketMLP -> FiLM(anchor) -> FiLM(event_global) -> CrossAttn(events)
      -> TemporalTransformer -> PredictionHead
    """

    def __init__(self, cfg, critical_indices):
        super().__init__()
        d = cfg.d_model

        # Market pathway
        self.market_mlp = MarketMLP(cfg.n_features, d)
        self.film_anchor = FiLMLayer(cfg.n_features, d)

        # Event pathway
        self.event_encoder = EventEncoder(cfg.qwen_hidden, d)
        self.film_event = FiLMLayer(d, d)
        self.cross_attn = EventMarketCrossAttention(d, cfg.n_heads, cfg.dropout)

        # Temporal modeling
        self.temporal_transformer = TemporalTransformer(
            d, cfg.n_heads, cfg.n_transformer_layers,
            cfg.ff_dim, cfg.dropout, cfg.lookback,
        )

        # Prediction
        self.prediction_head = PredictionHead(d, cfg.n_features, critical_indices)

    def forward(self, market_deltas, anchor_state, event_embeds, event_mask,
                target_deltas=None):
        """
        Args:
            market_deltas: (B, L, n_features)
            anchor_state: (B, n_features)
            event_embeds: (B, max_k, qwen_hidden)
            event_mask: (B, max_k) True = padded
            target_deltas: (B, H, n_features) or None
        Returns:
            predicted: (B, H, n_features)
            loss: scalar or None
        """
        # Market encoding
        h = self.market_mlp(market_deltas)          # (B, L, d)

        # FiLM Layer 1: anchor state modulation
        h = self.film_anchor(h, anchor_state)        # (B, L, d)

        # Event encoding
        e_global, e_reprs = self.event_encoder(event_embeds, event_mask)

        # FiLM Layer 2: event regime modulation
        h = self.film_event(h, e_global)             # (B, L, d)

        # Cross-attention: market attends to individual events
        h = self.cross_attn(h, e_reprs, event_mask)  # (B, L, d)

        # Temporal dependencies
        h = self.temporal_transformer(h)              # (B, L, d)

        # Predict future deltas
        predicted, loss = self.prediction_head(h, target_deltas)

        return predicted, loss

In [ ]:
# ============================================================
# Instantiate model and verify shapes
# ============================================================

model = FiLMWorldModel(cfg, critical_indices).to(DEVICE)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

# Forward pass sanity check with random data
with torch.no_grad():
    B = 4
    dummy = {
        "market_deltas": torch.randn(B, cfg.lookback, cfg.n_features).to(DEVICE),
        "anchor_state": torch.randn(B, cfg.n_features).to(DEVICE),
        "event_embeds": torch.randn(B, cfg.max_events_per_window, cfg.qwen_hidden).to(DEVICE),
        "event_mask": torch.ones(B, cfg.max_events_per_window, dtype=torch.bool).to(DEVICE),
        "target_deltas": torch.randn(B, cfg.horizon, cfg.n_features).to(DEVICE),
    }
    pred, loss = model(**dummy)
    print(f"Output shape: {pred.shape}")
    print(f"Loss: {loss.item():.6f}")

---
## Section C: Stage 1 Training (Frozen Qwen, Cached Embeddings)

In [ ]:
# ============================================================
# Training utilities
# ============================================================

def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0.0
    n_batches = 0

    for batch in loader:
        market_deltas = batch["market_deltas"].to(device)
        anchor_state = batch["anchor_state"].to(device)
        event_embeds = batch["event_embeds"].to(device)
        event_mask = batch["event_mask"].to(device)
        target_deltas = batch["target_deltas"].to(device)

        optimizer.zero_grad()
        _, loss = model(market_deltas, anchor_state, event_embeds, event_mask, target_deltas)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        n_batches += 1

    return total_loss / n_batches


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    total_loss = 0.0
    n_batches = 0

    for batch in loader:
        market_deltas = batch["market_deltas"].to(device)
        anchor_state = batch["anchor_state"].to(device)
        event_embeds = batch["event_embeds"].to(device)
        event_mask = batch["event_mask"].to(device)
        target_deltas = batch["target_deltas"].to(device)

        _, loss = model(market_deltas, anchor_state, event_embeds, event_mask, target_deltas)
        total_loss += loss.item()
        n_batches += 1

    return total_loss / n_batches

In [ ]:
# ============================================================
# Stage 1 training loop
# ============================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=cfg.stage1_lr,
    weight_decay=cfg.stage1_weight_decay,
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=cfg.stage1_epochs, eta_min=1e-6,
)

best_val_loss = float("inf")
train_losses = []
val_losses = []

print(f"Stage 1: {cfg.stage1_epochs} epochs, batch_size={cfg.stage1_batch_size}, lr={cfg.stage1_lr}")
print("=" * 60)

for epoch in range(1, cfg.stage1_epochs + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, DEVICE)
    val_loss = evaluate(model, val_loader, DEVICE)
    scheduler.step()

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "stage1_best.pt")

    if epoch % 10 == 0 or epoch == 1:
        lr = scheduler.get_last_lr()[0]
        print(f"Epoch {epoch:3d} | Train: {train_loss:.6f} | Val: {val_loss:.6f} | "
              f"Best Val: {best_val_loss:.6f} | LR: {lr:.2e}")

print(f"\nStage 1 complete. Best val loss: {best_val_loss:.6f}")

In [ ]:
# Plot training curves
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(train_losses, label="Train Loss", alpha=0.8)
ax.plot(val_losses, label="Val Loss", alpha=0.8)
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE Loss (critical features)")
ax.set_title("Stage 1: Training Curves")
ax.legend()
ax.set_yscale("log")
plt.tight_layout()
plt.show()

---
## Section D: Stage 2 Training (LoRA Fine-tuning)

Unfreeze Qwen3-4B LoRA adapters for joint fine-tuning.
Market pathway uses lower LR; LoRA uses 10x lower.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
from huggingface_hub import login

login(token=HF_token)

In [ ]:
# ============================================================
# Live Event Encoder with Qwen + LoRA
# ============================================================

class LiveEventEncoder(nn.Module):
    """Event encoder with live Qwen3-4B + LoRA inference.
    Replaces the cached EventEncoder for Stage 2."""

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg

        # Load Qwen with LoRA
        self.tokenizer = AutoTokenizer.from_pretrained(cfg.qwen_model_id)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        qwen = AutoModelForCausalLM.from_pretrained(
            cfg.qwen_model_id,
            torch_dtype=torch.bfloat16,
            device_map="auto",
        )
        # Enable gradient checkpointing for memory
        qwen.gradient_checkpointing_enable()

        lora_config = LoraConfig(
            r=cfg.lora_r,
            lora_alpha=cfg.lora_alpha,
            target_modules=list(cfg.lora_target_modules),
            lora_dropout=0.05,
            bias="none",
            task_type="CAUSAL_LM",
        )
        self.qwen = get_peft_model(qwen, lora_config)
        self.qwen.print_trainable_parameters()

        # Projection and no-event embed (will load from Stage 1)
        self.projection = nn.Linear(cfg.qwen_hidden, cfg.d_model)
        self.no_event_embed = nn.Parameter(torch.zeros(cfg.d_model))

    def encode_texts(self, texts, device):
        """Encode a list of event texts through Qwen -> projection."""
        if len(texts) == 0:
            return torch.zeros(0, self.cfg.d_model, device=device)

        inputs = self.tokenizer(
            texts, return_tensors="pt", padding=True,
            truncation=True, max_length=512,
        ).to(device)

        outputs = self.qwen(**inputs, output_hidden_states=True)
        last_hidden = outputs.hidden_states[-1]  # (n_texts, seq, hidden)

        # Last non-pad token
        attention_mask = inputs["attention_mask"]
        seq_lens = attention_mask.sum(dim=1) - 1  # (n_texts,)
        embeds = last_hidden[torch.arange(len(texts)), seq_lens]  # (n_texts, hidden)

        return self.projection(embeds.float())  # (n_texts, d_model)

    def forward(self, event_texts_batch, event_mask, device):
        """
        Args:
            event_texts_batch: list[list[str]] — per-sample event texts
            event_mask: (B, max_k) True = padded
            device: torch device
        Returns:
            e_global: (B, d_model)
            e_reprs: (B, max_k, d_model)
        """
        B = len(event_texts_batch)
        max_k = self.cfg.max_events_per_window
        d = self.cfg.d_model

        # Collect all unique texts, encode once, scatter back
        all_texts = []
        text_to_idx = {}
        sample_event_map = []  # list of list of indices into all_texts

        for sample_texts in event_texts_batch:
            indices = []
            for t in sample_texts:
                if t not in text_to_idx:
                    text_to_idx[t] = len(all_texts)
                    all_texts.append(t)
                indices.append(text_to_idx[t])
            sample_event_map.append(indices)

        # Encode all unique texts
        if all_texts:
            all_embeds = self.encode_texts(all_texts, device)  # (n_unique, d)
        else:
            all_embeds = torch.zeros(0, d, device=device)

        # Scatter into (B, max_k, d) tensor
        e_reprs = torch.zeros(B, max_k, d, device=device)
        for b, indices in enumerate(sample_event_map):
            for i, idx in enumerate(indices[:max_k]):
                e_reprs[b, i] = all_embeds[idx]

        # Global pool
        valid_mask = ~event_mask
        n_valid = valid_mask.sum(dim=1, keepdim=True).clamp(min=1)
        masked_reprs = e_reprs * valid_mask.unsqueeze(-1).float()
        e_global = masked_reprs.sum(dim=1) / n_valid.float()

        no_events = valid_mask.sum(dim=1) == 0
        if no_events.any():
            e_global[no_events] = self.no_event_embed.unsqueeze(0)

        return e_global, e_reprs

In [ ]:
# ============================================================
# Stage 2 World Model (swaps in LiveEventEncoder)
# ============================================================

class FiLMWorldModelStage2(nn.Module):
    """Stage 2 variant with live Qwen event encoding."""

    def __init__(self, cfg, critical_indices):
        super().__init__()
        d = cfg.d_model

        self.market_mlp = MarketMLP(cfg.n_features, d)
        self.film_anchor = FiLMLayer(cfg.n_features, d)
        self.live_event_encoder = LiveEventEncoder(cfg)
        self.film_event = FiLMLayer(d, d)
        self.cross_attn = EventMarketCrossAttention(d, cfg.n_heads, cfg.dropout)
        self.temporal_transformer = TemporalTransformer(
            d, cfg.n_heads, cfg.n_transformer_layers,
            cfg.ff_dim, cfg.dropout, cfg.lookback,
        )
        self.prediction_head = PredictionHead(d, cfg.n_features, critical_indices)

    def load_stage1_weights(self, stage1_model):
        """Transfer weights from Stage 1 model."""
        self.market_mlp.load_state_dict(stage1_model.market_mlp.state_dict())
        self.film_anchor.load_state_dict(stage1_model.film_anchor.state_dict())
        self.film_event.load_state_dict(stage1_model.film_event.state_dict())
        self.cross_attn.load_state_dict(stage1_model.cross_attn.state_dict())
        self.temporal_transformer.load_state_dict(stage1_model.temporal_transformer.state_dict())
        self.prediction_head.load_state_dict(stage1_model.prediction_head.state_dict())
        # Transfer event encoder projection + no_event_embed
        self.live_event_encoder.projection.load_state_dict(
            stage1_model.event_encoder.projection.state_dict()
        )
        self.live_event_encoder.no_event_embed.data.copy_(
            stage1_model.event_encoder.no_event_embed.data
        )
        print("Stage 1 weights loaded into Stage 2 model.")

    def forward(self, market_deltas, anchor_state, event_texts_batch, event_mask,
                target_deltas=None):
        h = self.market_mlp(market_deltas)
        h = self.film_anchor(h, anchor_state)

        e_global, e_reprs = self.live_event_encoder(
            event_texts_batch, event_mask, market_deltas.device,
        )

        h = self.film_event(h, e_global)
        h = self.cross_attn(h, e_reprs, event_mask)
        h = self.temporal_transformer(h)
        predicted, loss = self.prediction_head(h, target_deltas)

        return predicted, loss

In [ ]:
# ============================================================
# Stage 2 Dataset (returns event texts instead of embeddings)
# ============================================================

# Load pre-formatted event texts from Notebook 1
events_formatted_df = pd.read_csv(
    f"{cfg.data_dir}/events_with_formatted_text.csv"
)
events_formatted_df["event_time"] = pd.to_datetime(events_formatted_df["event_time"])
event_formatted_texts = events_formatted_df["formatted_text"].tolist()


class CommodityEventDatasetLive(Dataset):
    """Stage 2 dataset that returns event text strings."""

    def __init__(self, deltas_df, raw_df, events_df, event_texts, cfg,
                 start_date, end_date):
        self.cfg = cfg
        self.event_texts = event_texts

        mask = (deltas_df.index >= start_date) & (deltas_df.index <= end_date)
        self.deltas = deltas_df[mask].values.astype(np.float32)
        self.dates = deltas_df[mask].index

        raw_mask = (raw_df.index >= start_date) & (raw_df.index <= end_date)
        self.raw_values = raw_df[raw_mask].values.astype(np.float32)
        self.raw_dates = raw_df[raw_mask].index

        total_needed = cfg.lookback + cfg.horizon
        self.n_samples = max(0, len(self.deltas) - total_needed + 1)
        print(f"Live Dataset [{start_date} to {end_date}]: {self.n_samples} samples")

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        L = self.cfg.lookback
        H = self.cfg.horizon

        market_deltas = torch.from_numpy(self.deltas[idx : idx + L])
        target_deltas = torch.from_numpy(self.deltas[idx + L : idx + L + H])

        anchor_date = self.dates[idx + L - 1]
        raw_idx = self.raw_dates.get_loc(anchor_date)
        anchor_state = torch.from_numpy(self.raw_values[raw_idx])

        window_start = self.dates[idx]
        window_end = self.dates[idx + L - 1]
        event_indices = find_events_in_window(window_start, window_end)
        event_indices = event_indices[: self.cfg.max_events_per_window]
        n_events = len(event_indices)

        # Get formatted texts
        texts = [self.event_texts[i] for i in event_indices]

        # Event mask
        max_k = self.cfg.max_events_per_window
        event_mask = torch.ones(max_k, dtype=torch.bool)
        event_mask[:n_events] = False

        return {
            "market_deltas": market_deltas,
            "anchor_state": anchor_state,
            "target_deltas": target_deltas,
            "event_texts": texts,
            "event_mask": event_mask,
        }


def collate_fn_live(batch):
    return {
        "market_deltas": torch.stack([s["market_deltas"] for s in batch]),
        "anchor_state": torch.stack([s["anchor_state"] for s in batch]),
        "target_deltas": torch.stack([s["target_deltas"] for s in batch]),
        "event_texts": [s["event_texts"] for s in batch],  # list[list[str]]
        "event_mask": torch.stack([s["event_mask"] for s in batch]),
    }

In [ ]:
# ============================================================
# Initialize Stage 2
# ============================================================

# Load best Stage 1 model
model.load_state_dict(torch.load("stage1_best.pt", weights_only=True))
model.to(DEVICE)
print("Stage 1 best model loaded.")

# Create Stage 2 model and transfer weights
model_s2 = FiLMWorldModelStage2(cfg, critical_indices)
model_s2.load_stage1_weights(model)

# Free Stage 1 model memory
del model
torch.cuda.empty_cache()

# Create live datasets
train_ds_live = CommodityEventDatasetLive(
    deltas_df, market_df, events_df, event_formatted_texts, cfg,
    cfg.train_start, cfg.train_end,
)
val_ds_live = CommodityEventDatasetLive(
    deltas_df, market_df, events_df, event_formatted_texts, cfg,
    cfg.val_start, cfg.val_end,
)

train_loader_s2 = DataLoader(
    train_ds_live, batch_size=cfg.stage2_batch_size, shuffle=True,
    collate_fn=collate_fn_live, num_workers=0, pin_memory=True,
)
val_loader_s2 = DataLoader(
    val_ds_live, batch_size=cfg.stage2_batch_size, shuffle=False,
    collate_fn=collate_fn_live, num_workers=0, pin_memory=True,
)

In [ ]:
# ============================================================
# Stage 2 training loop with gradient accumulation
# ============================================================

# Separate param groups: market pathway vs LoRA
lora_params = []
market_params = []
for name, param in model_s2.named_parameters():
    if not param.requires_grad:
        continue
    if "qwen" in name and "lora" in name:
        lora_params.append(param)
    else:
        market_params.append(param)

print(f"Market params: {sum(p.numel() for p in market_params):,}")
print(f"LoRA params: {sum(p.numel() for p in lora_params):,}")

optimizer_s2 = torch.optim.AdamW([
    {"params": market_params, "lr": cfg.stage2_lr_market},
    {"params": lora_params, "lr": cfg.stage2_lr_lora},
], weight_decay=cfg.stage2_weight_decay)

scheduler_s2 = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_s2, T_max=cfg.stage2_epochs, eta_min=1e-7,
)

In [ ]:
def train_one_epoch_stage2(model, loader, optimizer, device, grad_accum):
    model.train()
    total_loss = 0.0
    n_batches = 0
    optimizer.zero_grad()

    for step, batch in enumerate(loader):
        market_deltas = batch["market_deltas"].to(device)
        anchor_state = batch["anchor_state"].to(device)
        event_texts = batch["event_texts"]  # list[list[str]]
        event_mask = batch["event_mask"].to(device)
        target_deltas = batch["target_deltas"].to(device)

        _, loss = model(market_deltas, anchor_state, event_texts, event_mask, target_deltas)
        loss = loss / grad_accum
        loss.backward()

        if (step + 1) % grad_accum == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad()

        total_loss += loss.item() * grad_accum
        n_batches += 1

    # Handle remaining gradients
    if n_batches % grad_accum != 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        optimizer.zero_grad()

    return total_loss / n_batches


@torch.no_grad()
def evaluate_stage2(model, loader, device):
    model.eval()
    total_loss = 0.0
    n_batches = 0

    for batch in loader:
        market_deltas = batch["market_deltas"].to(device)
        anchor_state = batch["anchor_state"].to(device)
        event_texts = batch["event_texts"]
        event_mask = batch["event_mask"].to(device)
        target_deltas = batch["target_deltas"].to(device)

        _, loss = model(market_deltas, anchor_state, event_texts, event_mask, target_deltas)
        total_loss += loss.item()
        n_batches += 1

    return total_loss / n_batches

In [ ]:
best_val_loss_s2 = float("inf")
train_losses_s2 = []
val_losses_s2 = []

print(f"Stage 2: {cfg.stage2_epochs} epochs, batch_size={cfg.stage2_batch_size}, "
      f"grad_accum={cfg.stage2_grad_accum}")
print(f"LR: market={cfg.stage2_lr_market}, LoRA={cfg.stage2_lr_lora}")
print("=" * 60)

for epoch in range(1, cfg.stage2_epochs + 1):
    train_loss = train_one_epoch_stage2(
        model_s2, train_loader_s2, optimizer_s2, DEVICE, cfg.stage2_grad_accum,
    )
    val_loss = evaluate_stage2(model_s2, val_loader_s2, DEVICE)
    scheduler_s2.step()

    train_losses_s2.append(train_loss)
    val_losses_s2.append(val_loss)

    if val_loss < best_val_loss_s2:
        best_val_loss_s2 = val_loss
        torch.save(model_s2.state_dict(), "stage2_best.pt")

    print(f"Epoch {epoch:3d} | Train: {train_loss:.6f} | Val: {val_loss:.6f} | "
          f"Best Val: {best_val_loss_s2:.6f}")

print(f"\nStage 2 complete. Best val loss: {best_val_loss_s2:.6f}")

In [ ]:
# Plot Stage 2 curves
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(train_losses_s2, label="Train Loss", alpha=0.8)
ax.plot(val_losses_s2, label="Val Loss", alpha=0.8)
ax.axhline(y=best_val_loss, color="gray", linestyle="--", alpha=0.5, label="Stage 1 Best Val")
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE Loss (critical features)")
ax.set_title("Stage 2: Training Curves")
ax.legend()
ax.set_yscale("log")
plt.tight_layout()
plt.show()